# 다중시점 기하학 (Multiview Geometry)

> 기준 COLMAP commit: `c004ddf51ee90bb70cbc2bb94a314c7f932febbe` (references/colmap)

여러 장의 2D 사진에서 3D 구조를 복원하는 **Structure from Motion (SfM)**의 수학적 기초를 설명합니다.

---

## 1. 전체 파이프라인 개요

### 1.1 COLMAP SfM 파이프라인

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        COLMAP SfM 파이프라인                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  [입력: 이미지들]                                                            │
│        │                                                                    │
│        ▼                                                                    │
│  ┌─────────────┐                                                            │
│  │ 1. Feature  │  SIFT로 각 이미지에서 특징점 추출                             │
│  │  Extraction │                                                            │
│  └──────┬──────┘                                                            │
│         │                                                                   │
│         ▼                                                                   │
│  ┌─────────────┐                                                            │
│  │ 2. Feature  │  이미지 쌍들 간의 특징점 매칭                                 │
│  │  Matching   │                                                            │
│  └──────┬──────┘                                                            │
│         │                                                                   │
│         ▼                                                                   │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │ 3. Two-View Geometry ★ 이 문서의 핵심                                │   │
│  │    ┌────────────────────────────────────────────────────────────┐   │   │
│  │    │  for each 이미지 쌍:                                        │   │   │
│  │    │    ① E/F 추정 (5-point 또는 7-point + RANSAC)              │   │   │
│  │    │    ② H 추정 (4-point + RANSAC)                             │   │   │
│  │    │    ③ inlier 비교 → 장면 타입 결정                           │   │   │
│  │    │       (H inlier > 80% of E inlier → 평면/파노라마)          │   │   │
│  │    │    ④ E에서 R, t 추출 (SVD 분해)                             │   │   │
│  │    │    ⑤ Triangulation으로 3D 점 복원 (SVD)                     │   │   │
│  │    └────────────────────────────────────────────────────────────┘   │   │
│  │    결과: 초기 카메라 포즈 + 초기 3D 점들 (정확도 낮음)               │   │
│  └──────┬──────────────────────────────────────────────────────────────┘   │
│         │                                                                   │
│         ▼                                                                   │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │ 4. Incremental Reconstruction (이미지를 하나씩 추가)                 │   │
│  │                                                                     │   │
│  │    초기: 2장으로 시작 → 3장 → 4장 → ... → N장 (점진적 확장)          │   │
│  │                                                                     │   │
│  │    ┌────────────────────────────────────────────────────────────┐   │   │
│  │    │  while 포즈가 결정 안 된 이미지 존재:                        │   │   │
│  │    │    ① 새 이미지 1장의 포즈 결정 (PnP)                        │   │   │
│  │    │       → 이미 알려진 3D 점들과의 대응으로 카메라 위치 계산    │   │   │
│  │    │    ② 새 3D 점 Triangulation                                 │   │   │
│  │    │    ③ Bundle Adjustment ← 전역 최적화 단계                   │   │   │
│  │    │       → 모든 카메라 포즈 + 모든 3D 점을 동시에 최적화        │   │   │
│  │    │       → Levenberg-Marquardt (비선형 최적화)                 │   │   │
│  │    └────────────────────────────────────────────────────────────┘   │   │
│  └──────┬──────────────────────────────────────────────────────────────┘   │
│         │                                                                   │
│         ▼                                                                   │
│  [출력: 카메라 포즈들 + Sparse Point Cloud]                                  │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```



### 1.2 핵심 질문과 해결 방법

```
질문: 여러 장의 2D 사진에서 3D 구조를 어떻게 복원할까?
```



**핵심 관찰:**
- 같은 3D 점 $\mathbf{X}$가 여러 카메라에서 관측됨
- 각 카메라에서의 2D 위치 $\mathbf{x}_1, \mathbf{x}_2, ...$ 를 알 수 있음
- 관측 간의 **기하학적 관계**로 3D 위치 추정 가능

### 1.3 SVD vs Bundle Adjustment: 역할 비교

| 구분 | SVD (Singular Value Decomposition) | Bundle Adjustment |
|------|-------------------------------------|-------------------|
| **성격** | 선형 대수 연산 | 비선형 최적화 |
| **역할** | **초기 추정값** 계산 | **정제(refinement)** |
| **사용 위치** | Two-View 단계 (E 추정, R/t 분해, 삼각측량) | Incremental 단계 (전역 최적화) |
| **계산량** | 빠름 (closed-form) | 느림 (반복 최적화) |
| **정확도** | 중간 (노이즈에 민감) | 높음 (노이즈 robust) |

**왜 둘 다 필요한가?**

```
SVD만 사용하면?
├── 문제: 각 이미지 쌍을 독립적으로 처리
├── 결과: 오차가 누적되어 전체 복원이 틀어짐
└── 비유: 퍼즐 조각을 대충 맞춘 상태

Bundle Adjustment를 적용하면?
├── 해결: 모든 관측을 동시에 고려하여 전역 최적화
├── 결과: 일관된 3D 복원
└── 비유: 퍼즐 전체를 보며 미세 조정
```



### 1.4 이 문서의 구성

이 문서는 **3번 단계 (Two-View Geometry)**의 수학적 기초를 상세히 설명합니다:

| 섹션 | 내용 | 핵심 개념 |
|------|------|-----------|
| **2장** | 카메라 모델 | 3D→2D 투영, $K$, $[R\|t]$ |
| **3장** | Epipolar 기하학 | 두 시점 간의 기하학적 제약 |
| **4장** | Essential vs Fundamental | E (calibrated), F (uncalibrated) |
| **5장** | E 추정 알고리듬 | 5-point + SVD + RANSAC |
| **6장** | Pose Recovery | E → R, t 분해 (SVD) |
| **7장** | Triangulation | 2D 대응점 → 3D 점 복원 |
| **8장** | Homography | 평면 장면의 특수 케이스 |
| **9장** | Configuration Type | 장면 분류 방법 |
| **10장** | Sampson Error | 오차 측정 방법 |
| **11장** | RANSAC | Outlier 처리 |

Bundle Adjustment의 상세 내용은 별도 문서에서 다룹니다.

---

## 2. 카메라 모델과 투영

### 2.1 Pinhole 카메라 모델

```
        3D World Point
              ●  X = (X, Y, Z)
              │
              │
    ──────────┼──────────  Image Plane (z = f)
              │
              ● Camera Center (0, 0, 0)
```



### 2.2 투영 방정식

$$
\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} \sim K [R | t] \begin{bmatrix} X \\ Y \\ Z \\ 1 \end{bmatrix}
$$

> **직관:** $\sim$는 "scale까지 동일"을 의미합니다. 3D 점이 카메라로부터 2배 멀리 있어도 같은 픽셀에 투영되기 때문에, 실제 계산에서는 마지막에 $z$로 나누어 정규화합니다.

- $K$: Camera Intrinsics (내부 파라미터) - 카메라 **내부** 특성 (렌즈 초점거리 등)
- $[R|t]$: Camera Extrinsics (외부 파라미터, 포즈) - 카메라가 **어디서 어느 방향**을 보는지

$$
K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}
$$

| 파라미터 | 의미 | 단위 |
|----------|------|------|
| $f_x, f_y$ | Focal length | 픽셀 |
| $c_x, c_y$ | Principal point | 픽셀 |

$$
[R|t] = \begin{bmatrix} r_{11} & r_{12} & r_{13} & t_x \\ r_{21} & r_{22} & r_{23} & t_y \\ r_{31} & r_{32} & r_{33} & t_z \end{bmatrix}
$$

| 파라미터 | 의미 | 단위 |
|----------|------|------|
| $R$ (3×3) | 회전 행렬 | - |
| $t$ (3×1) | 이동 벡터 | 미터 (또는 임의 스케일) |

**COLMAP의 카메라 모델:**

In [ ]:
// src/colmap/sensor/models.h
// COLMAP은 다양한 카메라 모델을 지원

MAKE_ENUM_CLASS_OVERLOAD_STREAM(CameraModelId,
    kSimplePinhole,   // f, cx, cy (3 params)
    kPinhole,         // fx, fy, cx, cy (4 params)
    kSimpleRadial,    // f, cx, cy, k (4 params) - 왜곡 1개
    kRadial,          // f, cx, cy, k1, k2 (5 params) - 왜곡 2개
    kOpenCV,          // fx, fy, cx, cy, k1, k2, p1, p2 (8 params)
    ...
);

// SimplePinhole: 왜곡 없는 가장 단순한 모델
struct SimplePinholeCameraModel {
  // 파라미터: f, cx, cy
  static constexpr size_t num_params = 3;
  static constexpr size_t num_focal_params = 1;  // f
  static constexpr size_t num_pp_params = 2;     // cx, cy
  static constexpr size_t num_extra_params = 0;  // 왜곡 없음
};

// Pinhole: fx, fy 분리 (비정방 픽셀)
struct PinholeCameraModel {
  // 파라미터: fx, fy, cx, cy
  static constexpr size_t num_params = 4;
};



> **3DGS에서:** `image_undistorter`로 왜곡을 제거하면 보통 `PINHOLE` 모델로 변환됩니다.

> **참고: 렌즈 왜곡**
> 실제 카메라는 렌즈 왜곡(radial/tangential distortion)이 있습니다. COLMAP은 왜곡 파라미터를 Bundle Adjustment에서 함께 추정하며, 3DGS 학습 전에 `image_undistorter`로 이미지를 보정합니다. 이 문서에서는 왜곡이 없는 이상적인 pinhole 모델을 가정합니다.

---

## 3. Epipolar Geometry (에피폴라 기하학)

두 시점 간의 기하학적 제약을 설명하는 핵심 개념입니다.

> **어원:** "Epipolar"는 그리스어 접두사 **"epi-"(위에, ~에 관한)**와 **"polar"(극점)**의 합성어입니다.
> - **Pole (극점)** = Epipole = 다른 카메라 중심이 투영된 점
> - **Epi-polar** = 극점에 관련된, 극점을 지나는
>
> 즉, "epipolar line"은 "epipole을 지나는 선"이라는 의미입니다. 모든 epipolar line은 해당 이미지의 epipole을 지나갑니다.

### 3.1 개념도

```
              ● X (3D point)
             /|\
            / | \
           /  |  \
          /   |   \
    ○────●────┼────●────○
   O₁   e₁    |    e₂   O₂
              |
           l₁ | l₂
          ────┼────
        image1  image2
```



| 용어 | 설명 |
|------|------|
| **Epipole** ($e_1, e_2$) | 다른 카메라의 중심이 투영된 점 |
| **Epipolar Line** ($l_1, l_2$) | 3D 점의 가능한 위치들이 투영된 선 |
| **Epipolar Plane** | $O_1$, $O_2$, $X$를 포함하는 평면 |

> **직관:** Image 1에서 점 $\mathbf{x}_1$을 관측했다면, 그 3D 점은 $O_1$과 $\mathbf{x}_1$을 잇는 **광선(ray) 위 어딘가**에 있습니다. 이 광선이 Image 2에 투영되면 **선(epipolar line)**이 됩니다. 따라서 대응점 $\mathbf{x}_2$는 반드시 이 선 위에 있어야 합니다!

### 3.2 Epipolar Constraint (에피폴라 제약)

**핵심 제약:**
$$
\mathbf{x}_2^T \mathbf{E} \mathbf{x}_1 = 0
$$

의미: 대응점 $\mathbf{x}_2$는 반드시 **epipolar line** 위에 있어야 한다.

이 제약은:
- 매칭의 **기하학적 검증**에 사용
- **outlier 제거** (잘못된 매칭 필터링)
- **카메라 포즈 추정**의 기초

### 3.3 COLMAP 코드: Epipole 계산

In [ ]:
// src/colmap/geometry/essential_matrix.cc
// E 행렬에서 epipole(다른 카메라의 중심이 투영된 점)을 계산
Eigen::Vector3d EpipoleFromEssentialMatrix(const Eigen::Matrix3d& E,
                                           const bool left_image) {
  Eigen::Vector3d e;
  if (left_image) {
    // E의 null space = 오른쪽 카메라 중심이 왼쪽 이미지에 투영된 위치
    Eigen::JacobiSVD<Eigen::Matrix3d> svd(E, Eigen::ComputeFullV);
    e = svd.matrixV().block<3, 1>(0, 2);  // V의 마지막 열
  } else {
    // E^T의 null space = 왼쪽 카메라 중심이 오른쪽 이미지에 투영된 위치
    Eigen::JacobiSVD<Eigen::Matrix3d> svd(E.transpose(), Eigen::ComputeFullV);
    e = svd.matrixV().block<3, 1>(0, 2);
  }
  return e;
}



> **수학적 배경:** Epipole은 E 행렬의 **null space**에 있습니다.
> - 왼쪽 epipole $e_1$: $\mathbf{E}^T e_1 = 0$
> - 오른쪽 epipole $e_2$: $\mathbf{E} e_2 = 0$
>
> SVD 분해에서 가장 작은 singular value에 해당하는 V의 열이 null space입니다.

---

## 4. Essential Matrix vs Fundamental Matrix

### 4.1 비교표

| | Essential Matrix ($\mathbf{E}$) | Fundamental Matrix ($\mathbf{F}$) |
|---|---------------------|------------------------|
| **제약식** | $\hat{\mathbf{x}}_2^T \mathbf{E} \hat{\mathbf{x}}_1 = 0$ | $\mathbf{x}_2^T \mathbf{F} \mathbf{x}_1 = 0$ |
| **좌표계** | 정규화 좌표 ($\hat{\mathbf{x}} = K^{-1}\mathbf{x}$) | 픽셀 좌표 |
| **K 필요** | ✅ 필요 | ❌ 불필요 |
| **자유도** | 5 | 7 |
| **최소 점 수** | 5점 | 7점 (또는 8점) |

> **직관:** E는 "카메라 내부 특성을 이미 알고 있을 때" 사용하고, F는 "카메라에 대해 아무것도 모를 때"도 사용할 수 있습니다. COLMAP은 EXIF에서 K를 추정하므로 보통 **E를 사용**합니다.

### 4.2 관계식

$$
\mathbf{E} = \mathbf{K}_2^T \mathbf{F} \mathbf{K}_1
$$

동일한 카메라 사용 시:
$$
\mathbf{E} = \mathbf{K}^T \mathbf{F} \mathbf{K}
$$

**COLMAP 코드:**

In [ ]:
// src/colmap/geometry/essential_matrix.cc

// F에서 E로 변환: E = K2^T * F * K1
Eigen::Matrix3d EssentialFromFundamentalMatrix(const Eigen::Matrix3d& K2,
                                               const Eigen::Matrix3d& F,
                                               const Eigen::Matrix3d& K1) {
  return K2.transpose() * F * K1;
}

// E에서 F로 변환: F = K2^(-T) * E * K1^(-1)
Eigen::Matrix3d FundamentalFromEssentialMatrix(const Eigen::Matrix3d& K2,
                                               const Eigen::Matrix3d& E,
                                               const Eigen::Matrix3d& K1) {
  return K2.transpose().inverse() * E * K1.inverse();
}



### 4.3 Essential Matrix의 구조

$$
\mathbf{E} = [\mathbf{t}]_\times \mathbf{R}
$$

여기서 $[\mathbf{t}]_\times$는 translation의 skew-symmetric matrix:

$$
[\mathbf{t}]_\times = \begin{bmatrix} 0 & -t_z & t_y \\ t_z & 0 & -t_x \\ -t_y & t_x & 0 \end{bmatrix}
$$

> **직관:** $[\mathbf{t}]_\times$는 **외적(cross product)을 행렬 곱으로 표현**한 것입니다.
> $$[\mathbf{t}]_\times \mathbf{v} = \mathbf{t} \times \mathbf{v}$$
> 즉, 어떤 벡터 $\mathbf{v}$에 이 행렬을 곱하면 $\mathbf{t}$와 $\mathbf{v}$의 외적이 됩니다.
>
> **왜 이렇게 표현하나?** 외적을 행렬 곱으로 바꾸면 선형대수 도구(SVD, 행렬 분해 등)를 사용할 수 있고, $E = [\mathbf{t}]_\times R$처럼 회전과 이동을 하나의 행렬로 결합할 수 있습니다.

**왜 $\mathbf{E} = [\mathbf{t}]_\times \mathbf{R}$ 형태인가?**

Epipolar constraint의 기하학적 의미에서 유도됩니다:

```
세 벡터가 같은 평면(Epipolar Plane)에 있어야 함:
1. t: 카메라1에서 카메라2로의 방향 (baseline)
2. x₁: 카메라1에서 3D 점 방향
3. x₂: 카메라2에서 3D 점 방향

         ● X (3D point)
        /|
       / |
    x₂/  |x₁  ← 이 세 벡터가 
     /   |      같은 평면에!
    /    |
   ○─────○
  cam2  cam1
     t →
```



**수학적 유도:**

1. 카메라1에서 본 방향 $\mathbf{x}_1$을 카메라2 좌표계로 변환: $\mathbf{R} \mathbf{x}_1$
2. 세 벡터 $\mathbf{x}_2$, $\mathbf{t}$, $\mathbf{R}\mathbf{x}_1$이 같은 평면 → **삼중곱(triple product) = 0**
3. 삼중곱: $\mathbf{x}_2 \cdot (\mathbf{t} \times \mathbf{R}\mathbf{x}_1) = 0$
4. 외적을 행렬로 표현: $\mathbf{t} \times \mathbf{v} = [\mathbf{t}]_\times \mathbf{v}$
5. 따라서: $\mathbf{x}_2^T [\mathbf{t}]_\times \mathbf{R} \mathbf{x}_1 = 0$

$$\therefore \mathbf{E} = [\mathbf{t}]_\times \mathbf{R}$$

| 성분 | 역할 |
|------|------|
| $\mathbf{R}$ | 카메라1 좌표계 → 카메라2 좌표계 변환 |
| $[\mathbf{t}]_\times$ | baseline과 같은 평면인지 확인 (외적으로 검사) |

**중요:** E는 두 카메라 간의 **상대 포즈 (R, t)**를 인코딩합니다!

**COLMAP 코드:**

In [ ]:
// src/colmap/geometry/essential_matrix.cc
// 상대 포즈(R, t)로부터 E 행렬 생성
Eigen::Matrix3d EssentialMatrixFromPose(const Rigid3d& cam2_from_cam1) {
  // E = [t]× * R
  // CrossProductMatrix(t)는 skew-symmetric matrix [t]×를 생성
  return CrossProductMatrix(cam2_from_cam1.translation().normalized()) *
         cam2_from_cam1.rotation().toRotationMatrix();
}

// src/colmap/geometry/pose.cc
// 외적을 행렬 곱으로 표현하는 skew-symmetric matrix 생성
Eigen::Matrix3d CrossProductMatrix(const Eigen::Vector3d& vector) {
  Eigen::Matrix3d matrix;
  matrix << 0, -vector(2), vector(1),
            vector(2), 0, -vector(0),
            -vector(1), vector(0), 0;
  return matrix;
}



---

## 5. Five-Point Algorithm (5점 알고리듬)

COLMAP의 기본 알고리듬입니다 (calibrated camera).

### 5.1 왜 5점인가?

- E는 5 자유도 (3 rotation + 2 translation 방향)
- 각 대응점이 1개의 제약 제공
- 최소 5개의 점 필요

> **직관:** E는 3×3 행렬(9개 원소)이지만, scale 자유도 1개와 내부 제약 3개($\text{rank}=2$, Essential matrix 조건)를 빼면 **5 자유도**만 남습니다. Translation은 **방향만** 알 수 있고 크기(scale)는 알 수 없습니다 - 이것이 SfM에서 "scale ambiguity"가 발생하는 이유입니다.

**Scale Ambiguity란?**

```
장면 A (실제 크기)              장면 B (10배 스케일)

○──1m──○                       ○────10m────○
cam1   cam2                     cam1        cam2
  \     /                         \        /
   \   /                           \      /
    \ /                             \    /
     ● 물체 (10m 앞)                  ● 물체 (100m 앞)

두 장면의 Essential Matrix E가 동일!
→ E만으로는 baseline이 1m인지 10m인지 알 수 없음
```



### 5.2 COLMAP 코드

In [ ]:
// src/colmap/estimators/solvers/essential_matrix.cc
void EssentialMatrixFivePointEstimator::Estimate(
    const std::vector<X_t>& cam_rays1,
    const std::vector<Y_t>& cam_rays2,
    std::vector<M_t>* models) {
  
  // Step 1: 제약 행렬 구성
  // x2^T * E * x1 = 0 → [x2 ⊗ x1]^T * vec(E) = 0
  Eigen::Matrix<double, Eigen::Dynamic, 9> Q(cam_rays1.size(), 9);
  for (size_t i = 0; i < cam_rays1.size(); ++i) {
    Q.row(i) << cam_rays2[i].x() * cam_rays1[i].transpose(),
                cam_rays2[i].y() * cam_rays1[i].transpose(),
                cam_rays2[i].z() * cam_rays1[i].transpose();
  }

  // Step 2: SVD로 null space 추출
  // Q * vec(E) = 0 을 만족하는 E를 찾기 위해 V의 마지막 4열 사용
  Eigen::Matrix<double, 9, 4> E;
  if (cam_rays1.size() == 5) {
    // 정확히 5점: QR 분해 사용 (더 빠름)
    E = Q.transpose().fullPivHouseholderQr().matrixQ().rightCols<4>();
  } else {
    // 5점 이상: SVD 사용
    const Eigen::JacobiSVD<Eigen::Matrix<double, Eigen::Dynamic, 9>> svd(
        Q, Eigen::ComputeFullV);
    E = svd.matrixV().rightCols<4>();  // null space basis (4차원)
  }

  // Step 3-5: 다항식 제약으로 E 복원
  // E의 제약: det(E) = 0, 2*E*E^T*E - trace(E*E^T)*E = 0
  // → 10차 다항식 풀이 후 실수 근에서 최종 E 복원
}



> **직관:** 5개 대응점은 9차원 공간(E의 9개 원소)에서 5개의 제약을 제공합니다. 따라서 해 공간은 4차원 null space입니다. SVD의 마지막 4개 right singular vector가 이 null space의 기저(basis)가 됩니다. 이후 Essential matrix의 내부 제약($\text{rank}=2$ 등)을 이용해 유일한 해를 찾습니다.

---

## 6. Pose Recovery: E에서 R, t 추출

Essential Matrix를 SVD 분해하여 카메라 포즈를 얻습니다.

### 6.1 SVD 분해

$$
\mathbf{E} = \mathbf{U} \Sigma \mathbf{V}^T = \mathbf{U} \begin{bmatrix} \sigma & 0 & 0 \\ 0 & \sigma & 0 \\ 0 & 0 & 0 \end{bmatrix} \mathbf{V}^T
$$

> **직관:** E의 singular value가 $(\sigma, \sigma, 0)$ 형태인 이유는 E가 **rank 2** 행렬이기 때문입니다. 이는 $E = [\mathbf{t}]_\times R$에서 skew-symmetric matrix $[\mathbf{t}]_\times$가 항상 rank 2이기 때문입니다.

**Rank란?**

| Rank | 의미 | 예시 |
|------|------|------|
| **Rank 3** (Full rank) | 3×3 행렬의 모든 열/행이 독립 | 일반적인 회전 행렬 $R$ |
| **Rank 2** | 하나의 열/행이 다른 것들의 조합 | Essential Matrix $E$ |
| **Rank 1** | 두 개의 열/행이 다른 것의 조합 | 벡터의 외적 $\mathbf{u}\mathbf{v}^T$ |

> **직관:** Rank는 행렬이 "얼마나 많은 독립적인 정보"를 담고 있는지를 나타냅니다.
> - Rank 3 행렬: 3D 공간 전체를 표현 가능
> - Rank 2 행렬: 3D 공간 중 **평면**만 표현 가능 (하나의 차원 손실)
> - Rank 1 행렬: 3D 공간 중 **직선**만 표현 가능
>
> E가 rank 2인 것은 **translation 방향으로 정보가 손실**되기 때문입니다. (scale을 알 수 없음)

### 6.2 COLMAP 코드

In [ ]:
// src/colmap/geometry/essential_matrix.cc
void DecomposeEssentialMatrix(const Eigen::Matrix3d& E,
                              Eigen::Matrix3d* R1,
                              Eigen::Matrix3d* R2,
                              Eigen::Vector3d* t) {
  Eigen::JacobiSVD<Eigen::Matrix3d> svd(
      E, Eigen::ComputeFullU | Eigen::ComputeFullV);
  Eigen::Matrix3d U = svd.matrixU();
  Eigen::Matrix3d V = svd.matrixV().transpose();

  // det(R) = 1 보장 (reflection 제거)
  if (U.determinant() < 0) U *= -1;
  if (V.determinant() < 0) V *= -1;

  Eigen::Matrix3d W;
  W << 0, 1, 0, -1, 0, 0, 0, 0, 1;

  *R1 = U * W * V;              // 첫 번째 회전 후보
  *R2 = U * W.transpose() * V;  // 두 번째 회전 후보
  *t = U.col(2).normalized();   // translation: U의 3번째 열
}



### 6.3 Cheirality Test로 올바른 포즈 선택

In [ ]:
// src/colmap/geometry/essential_matrix.cc
// 4가지 후보 중 올바른 (R, t)를 선택
void PoseFromEssentialMatrix(const Eigen::Matrix3d& E,
                             const std::vector<Eigen::Vector3d>& cam_rays1,
                             const std::vector<Eigen::Vector3d>& cam_rays2,
                             Rigid3d* cam2_from_cam1,
                             std::vector<Eigen::Vector3d>* points3D) {
  Eigen::Matrix3d R1, R2;
  Eigen::Vector3d t;
  DecomposeEssentialMatrix(E, &R1, &R2, &t);

  // 4가지 (R, t) 조합 생성
  const std::array<Rigid3d, 4> candidates{{
    Rigid3d(R1, t), Rigid3d(R2, t),
    Rigid3d(R1, -t), Rigid3d(R2, -t)
  }};

  // Cheirality Test: 가장 많은 점이 두 카메라 "앞"에 있는 조합 선택
  for (const auto& pose : candidates) {
    std::vector<Eigen::Vector3d> tentative_points3D;
    CheckCheirality(pose, cam_rays1, cam_rays2, &tentative_points3D);
    if (tentative_points3D.size() >= points3D->size()) {
      *cam2_from_cam1 = pose;
      std::swap(*points3D, tentative_points3D);
    }
  }
}



> **Cheirality Test:** 3D 점이 두 카메라 **앞**에 있어야 유효합니다. 4가지 후보 중 가장 많은 점이 양쪽 카메라 앞에 있는 조합이 정답입니다.

---

## 7. Triangulation (삼각측량)

두 시점에서 관측된 2D 대응점으로부터 3D 점을 복원합니다.

> **Pose Recovery(6장)와의 관계:**
> | | Pose Recovery (6장) | Triangulation (7장) |
> |---|---|---|
> | **입력** | Essential Matrix $\mathbf{E}$ | 카메라 포즈 $(\mathbf{R}, \mathbf{t})$ + 2D 대응점 |
> | **출력** | 카메라 상대 포즈 $(\mathbf{R}, \mathbf{t})$ | 3D 점 좌표 $\mathbf{X}$ |
> | **질문** | "카메라가 어디에 있는가?" | "3D 점이 어디에 있는가?" |
>
> 순서: E → [Pose Recovery] → (R, t) → [Triangulation] → 3D Point
>
> Pose Recovery가 먼저 필요한 이유: Triangulation은 두 카메라에서 쏜 **광선의 교점**을 찾는 것인데, 광선을 정의하려면 카메라 포즈 (R, t)가 필수입니다.

### 7.1 기본 원리

```
              ● X (unknown 3D point)
             /\
            /  \
       ray1/    \ray2
          /      \
         /        \
        ○──────────○
       cam1       cam2
        ↓          ↓
       x₁         x₂  (known 2D points)
```



**핵심 아이디어:** 두 카메라에서 쏜 광선(ray)의 교점이 3D 점

- 이상적인 경우: 두 광선이 정확히 한 점에서 만남
- 현실: 노이즈로 인해 광선이 만나지 않음 → **최소자승해** 필요

### 7.2 수학적 정식화

각 카메라의 투영 방정식:
$$\mathbf{x}_1 \sim P_1 \mathbf{X}, \quad \mathbf{x}_2 \sim P_2 \mathbf{X}$$

여기서 $P_i = K_i[R_i|t_i]$는 카메라 투영 행렬입니다.

외적을 이용한 선형화 (DLT 방식):
$$\mathbf{x} \times (P\mathbf{X}) = 0$$

이를 전개하면 $A\mathbf{X} = 0$ 형태의 선형 시스템:

$$
A = \begin{bmatrix}
x_1 p_1^{3T} - p_1^{1T} \\
y_1 p_1^{3T} - p_1^{2T} \\
x_2 p_2^{3T} - p_2^{1T} \\
y_2 p_2^{3T} - p_2^{2T}
\end{bmatrix}
$$

여기서 $p_i^{jT}$는 $P_i$의 j번째 행입니다.

### 7.3 SVD를 이용한 해법

$A\mathbf{X} = 0$의 최소자승해는 $A$의 **SVD에서 가장 작은 singular value에 해당하는 right singular vector**:

$$A = U \Sigma V^T$$

$\mathbf{X} = V_{:,4}$ (마지막 열)

> **직관:** $A\mathbf{X} = 0$을 정확히 만족하는 해가 없을 때, $\|A\mathbf{X}\|$를 최소화하는 단위 벡터 $\mathbf{X}$를 찾습니다. SVD의 마지막 right singular vector가 이 역할을 합니다.

### 7.4 COLMAP 코드

In [ ]:
// src/colmap/geometry/triangulation.cc
Eigen::Vector3d TriangulatePoint(const Eigen::Matrix3x4d& cam1_from_world,
                                  const Eigen::Matrix3x4d& cam2_from_world,
                                  const Eigen::Vector2d& point1,
                                  const Eigen::Vector2d& point2) {
  Eigen::Matrix4d A;
  
  // 각 관측에서 2개의 방정식
  A.row(0) = point1(0) * cam1_from_world.row(2) - cam1_from_world.row(0);
  A.row(1) = point1(1) * cam1_from_world.row(2) - cam1_from_world.row(1);
  A.row(2) = point2(0) * cam2_from_world.row(2) - cam2_from_world.row(0);
  A.row(3) = point2(1) * cam2_from_world.row(2) - cam2_from_world.row(1);

  // SVD로 null space 찾기
  Eigen::JacobiSVD<Eigen::Matrix4d> svd(A, Eigen::ComputeFullV);
  return svd.matrixV().col(3).hnormalized();  // homogeneous → 3D
}



### 7.5 Triangulation 품질

**Triangulation Angle (삼각측량 각도):**

```
              ● X
             /\
            /θ \
           /    \
          ○──────○
        cam1    cam2
```



| 각도 θ | 품질 | 이유 |
|--------|------|------|
| 작음 (< 2°) | ❌ 나쁨 | 광선이 거의 평행 → 깊이 추정 불안정 |
| 중간 (5-30°) | ✅ 좋음 | 적절한 baseline |
| 큼 (> 60°) | ⚠️ 주의 | 매칭이 어려울 수 있음 |

> **직관:** 두 눈(카메라)이 너무 가까우면 거리감을 느끼기 어렵고, 너무 멀면 같은 물체를 인식하기 어렵습니다.

COLMAP은 `min_triangulation_angle` (기본값 1.5°) 이하의 점은 필터링합니다.

---

## 8. Homography Matrix (호모그래피)

### 8.1 정의

평면 위의 점들 또는 순수 회전 시 두 이미지 간의 변환:

$$
\mathbf{x}_2 \sim \mathbf{H} \mathbf{x}_1
$$

**H의 수학적 구조:**

| 상황 | 수식 | 설명 |
|------|------|------|
| **평면 장면** | $\mathbf{H} = \mathbf{K}_2 \left( \mathbf{R} - \frac{\mathbf{t}\mathbf{n}^T}{d} \right) \mathbf{K}_1^{-1}$ | R, t + 평면 정보 혼합 |
| **순수 회전** | $\mathbf{H} = \mathbf{K}_2 \mathbf{R} \mathbf{K}_1^{-1}$ | t=0일 때 (회전만) |

- $\mathbf{n}$: 평면의 법선 벡터
- $d$: 카메라에서 평면까지 거리

> **주의:** H는 순수 회전 행렬이 **아닙니다!** 평면 장면에서는 회전, 이동, 평면 정보가 혼합되어 있고, 순수 회전일 때만 (calibration 제외) 회전만 포함합니다.

### 8.2 H 추정: DLT + SVD

**왜 4점인가?**
- H는 3×3 = 9개 원소, scale 자유도 1개 제외 → **8 자유도**
- 각 대응점이 2개 방정식 제공 → **최소 4점** 필요

**DLT (Direct Linear Transform) 방식:**

각 대응점 $(\mathbf{x}_1, \mathbf{x}_2)$에서 $\mathbf{x}_2 \times (\mathbf{H}\mathbf{x}_1) = 0$을 전개하면:

$$A \cdot \text{vec}(\mathbf{H}) = 0$$

SVD로 해를 구함: $A = U \Sigma V^T$ → $\mathbf{H}$ = $V$의 마지막 열

**COLMAP 코드:**

In [ ]:
// src/colmap/estimators/solvers/homography_matrix.cc
void HomographyMatrixEstimator::Estimate(const std::vector<X_t>& points1,
                                         const std::vector<Y_t>& points2,
                                         std::vector<M_t>* models) {
  const size_t num_points = points1.size();

  // Step 1: 제약 행렬 A 구성 (각 점이 2개 행 생성)
  Eigen::Matrix<double, Eigen::Dynamic, 9> A(2 * num_points, 9);
  for (size_t i = 0; i < num_points; ++i) {
    // x2 × (H·x1) = 0 전개
    A.block<1, 3>(2 * i, 0) = points1[i].transpose().homogeneous();
    A.block<1, 3>(2 * i, 3).setZero();
    A.block<1, 3>(2 * i, 6) =
        -points2[i].x() * points1[i].transpose().homogeneous();
    A.block<1, 3>(2 * i + 1, 0).setZero();
    A.block<1, 3>(2 * i + 1, 3) = points1[i].transpose().homogeneous();
    A.block<1, 3>(2 * i + 1, 6) =
        -points2[i].y() * points1[i].transpose().homogeneous();
  }

  Eigen::Matrix3d H;
  if (num_points == 4) {
    // 정확히 4점: LU 분해 사용 (더 빠름)
    const Eigen::Matrix<double, 9, 1> h = A.block<8, 8>(0, 0)
                                              .partialPivLu()
                                              .solve(-A.block<8, 1>(0, 8))
                                              .homogeneous();
    H = Eigen::Map<const Eigen::Matrix3d>(h.data()).transpose();
  } else {
    // Step 2: SVD로 null space 찾기
    Eigen::JacobiSVD<Eigen::Matrix<double, Eigen::Dynamic, 9>> svd(
        A, Eigen::ComputeFullV);
    const Eigen::VectorXd nullspace = svd.matrixV().col(8);  // 마지막 열
    H = Eigen::Map<const Eigen::Matrix3d>(nullspace.data()).transpose();
  }

  models->push_back(H);
}



> **직관:** E/F 추정과 매우 유사합니다! 차이점은 H는 내부 제약(rank=2 등)이 없어서 SVD로 바로 해를 얻을 수 있다는 것입니다.

**E, F, H 비교:**

| | Essential (E) | Fundamental (F) | Homography (H) |
|---|---|---|---|
| **자유도** | 5 | 7 | 8 |
| **최소 점** | 5점 | 7점 | 4점 |
| **해법** | SVD + 다항식 | SVD | SVD (DLT) |
| **추가 제약** | rank=2, 내부 제약 | rank=2 | 없음 |

### 8.3 E vs H: 어떤 것을 사용할지 어떻게 결정하나?

**핵심:** 사전에 알 수 없으므로 **둘 다 추정하고 비교**합니다.

```
┌─────────────────────────────────────────────────┐
│   Feature Matches (대응점들)                     │
└─────────────────────────────────────────────────┘
          │                    │
          ▼                    ▼
   ┌─────────────┐      ┌─────────────┐
   │ E 추정      │      │ H 추정      │
   │ (5-point)   │      │ (4-point)   │
   └─────────────┘      └─────────────┘
          │                    │
          ▼                    ▼
    E inlier 개수         H inlier 개수
          │                    │
          └────────┬───────────┘
                   ▼
            ┌──────────────┐
            │ H/E > 0.8?   │
            └──────────────┘
           No │         │ Yes
              ▼         ▼
         일반 3D     평면/파노라마
         → E 사용    → 주의 필요
```



**왜 이렇게 동작하나?**

| 장면 유형 | E의 inlier | H의 inlier | H/E 비율 | 결과 |
|-----------|------------|------------|----------|------|
| **일반 3D** | 높음 | 낮음 | < 0.8 | E 사용 |
| **평면 장면** | 높음 (불안정) | 높음 | > 0.8 | 평면 감지 |
| **순수 회전** | 낮음 | 높음 | >> 0.8 | 파노라마 감지 |

> **직관:** 일반 3D 장면에서는 점들이 공간에 분포하므로 H로는 잘 맞지 않습니다. 반면 평면/회전 케이스에서는 H가 변환을 완벽히 설명할 수 있어 inlier가 많아집니다.

### 8.4 왜 Degenerate Case인가?

- **평면 장면:** 모든 점이 한 평면 위 → E/F가 무한히 많은 해를 가짐
- **순수 회전:** baseline = 0 → 삼각측량 불가능 (모든 광선이 평행)

> **직관:** E/F는 **baseline(카메라 간 거리)**이 있어야 의미가 있습니다. 이런 degenerate case에서는 H만으로 변환을 완벽히 설명할 수 있습니다.

```
                    일반 3D 장면              평면 장면 / 파노라마
                         ↓                           ↓
                  E/F 사용 가능              E/F 불안정 → H로 감지
```



### 8.5 COLMAP의 모델 선택 전략

In [ ]:
// src/colmap/estimators/two_view_geometry.cc
// E와 H를 모두 추정하고 inlier 비율 비교

const double H_E_inlier_ratio =
    static_cast<double>(H_report.support.num_inliers) /
    E_report.support.num_inliers;

if (H_E_inlier_ratio > options.max_H_inlier_ratio) {  // 기본값: 0.8
  geometry.config = TwoViewGeometry::ConfigurationType::PLANAR_OR_PANORAMIC;
} else {
  geometry.config = TwoViewGeometry::ConfigurationType::CALIBRATED;
}



**판단 기준:**
- H의 inlier가 E의 80% 이상이면 → 평면/파노라마로 판단
- 그렇지 않으면 → 일반 3D 장면

---

## 9. Configuration Type (장면 분류)

COLMAP은 two-view geometry를 분석하여 장면 유형을 분류합니다.

In [ ]:
// src/colmap/scene/two_view_geometry.h
enum class ConfigurationType {
  UNDEFINED,            // 미정의
  DEGENERATE,           // 추정 실패
  CALIBRATED,           // E 행렬 사용 (일반 3D, K 알려짐)
  UNCALIBRATED,         // F 행렬 사용 (K 모름)
  PLANAR,               // 평면 장면
  PANORAMIC,            // 순수 회전 (카메라 이동 없음)
  PLANAR_OR_PANORAMIC,  // 평면 또는 파노라마
  WATERMARK,            // 워터마크 감지
  MULTIPLE              // 여러 독립적인 움직임
};



### 3DGS에서의 의미

- `CALIBRATED` → 정상적인 SfM 진행 가능
- `PLANAR_OR_PANORAMIC` → 초기화 문제 발생 가능 (baseline 부족)
- `DEGENERATE` → 매칭 실패

---

## 10. Sampson Error (오차 측정)

기하학적 오차를 측정하는 대수적 근사.

### 10.1 정의

$$
d_{Sampson}^2 = \frac{(\mathbf{x}_2^T \mathbf{E} \mathbf{x}_1)^2}{(\mathbf{E}\mathbf{x}_1)_1^2 + (\mathbf{E}\mathbf{x}_1)_2^2 + (\mathbf{E}^T\mathbf{x}_2)_1^2 + (\mathbf{E}^T\mathbf{x}_2)_2^2}
$$

> **주의:** 이 수식은 **하나의 대응점 쌍** $(\mathbf{x}_1, \mathbf{x}_2)$에 대한 오차입니다.
> - $\mathbf{x}_1$: 이미지 1의 한 점
> - $\mathbf{x}_2$: 이미지 2의 대응점 (같은 3D 점의 투영)
>
> RANSAC에서는 이 수식을 **모든 대응점 쌍에 반복 적용**하여 inlier를 카운트합니다.

### 10.2 직관적 이해

- 분자: epipolar constraint 위반 정도 (0이면 완벽히 만족)
- 분모: 정규화 요소 (gradient 크기)

> **직관:** 이상적으로 $\mathbf{x}_2^T \mathbf{E} \mathbf{x}_1 = 0$이어야 하지만, 노이즈로 인해 0이 아닌 값이 됩니다. Sampson error는 이 위반을 **픽셀 단위의 거리**로 변환해줍니다.
>
> 예: Sampson error = 4이면, 대응점이 epipolar line에서 약 **2픽셀** 떨어져 있다는 의미입니다.

Sampson error는 **geometric error의 1차 근사**로, 계산이 빠르면서도 기하학적 의미가 있습니다.

### 10.3 RANSAC에서의 역할

**핵심 질문:** RANSAC에서 랜덤 5점으로 E를 추정했는데, 왜 **나머지 모든 점**에 대해 뭔가를 계산하는가?

**답:** 랜덤 5점에 outlier가 포함되어 있으면 추정된 E가 틀립니다. 이를 검증하기 위해 **모든 점**에 대해 Sampson Error를 계산합니다.

```
RANSAC 한 번의 반복:

1. 랜덤 5점 선택        →  이 5점으로 E 추정
                             │
2. 나머지 모든 점에 대해  ←──┘
   Sampson Error 계산
                             │
3. Error < threshold?   ←──┘
   ├── Yes → inlier (이 E가 잘 설명하는 점)
   └── No  → outlier (이 E로 설명 안 되는 점)
                             │
4. inlier 개수 카운트   ←──┘
   → 가장 많은 inlier를 가진 E가 "최고의 모델"
```



**왜 이렇게 하나?**

| 상황 | 랜덤 5점 | 추정된 E | 나머지 점들의 반응 |
|------|----------|----------|--------------------|
| **운 좋음** | 모두 inlier | ✅ 정확 | 대부분 낮은 error → inlier 많음 |
| **운 나쁨** | outlier 포함 | ❌ 틀림 | 대부분 높은 error → inlier 적음 |

> **직관:** 5점만으로는 E가 맞는지 알 수 없습니다. "좋은 E"라면 **다른 점들도** 이 E를 잘 만족해야 합니다. Sampson Error는 각 점이 E를 얼마나 잘 만족하는지 측정하는 **검증 도구**입니다.

**Threshold 설정:**

COLMAP 기본값: `max_error = 4.0` (픽셀²)
- Sampson Error < 4.0 → inlier
- Sampson Error ≥ 4.0 → outlier

이는 대략 **2픽셀** 이내의 오차를 허용한다는 의미입니다.

### 10.4 COLMAP 코드

In [ ]:
// src/colmap/geometry/essential_matrix.cc
double ComputeSquaredSampsonError(const Eigen::Vector3d& ray1,
                                  const Eigen::Vector3d& ray2,
                                  const Eigen::Matrix3d& E) {
  const Eigen::Vector3d epipolar_line1 = E * ray1;
  const double num = ray2.dot(epipolar_line1);  // x2^T * E * x1
  
  const Eigen::Vector4d denom(
    ray2.dot(E.col(0)),   // (E^T x2)_1
    ray2.dot(E.col(1)),   // (E^T x2)_2
    epipolar_line1.x(),   // (E x1)_1
    epipolar_line1.y()    // (E x1)_2
  );
  
  return num * num / denom.squaredNorm();
}



---

## 11. Two-View Geometry 추정 흐름

### 11.1 전체 흐름

```
Feature Matches
       │
       ▼
┌──────────────────────────────────────────┐
│  Calibrated 여부 확인                     │
│  (camera.has_prior_focal_length)         │
└──────────────────────────────────────────┘
       │
       ├── Calibrated ──────────────────────┐
       │                                    │
       ▼                                    ▼
┌─────────────────┐               ┌─────────────────┐
│ E 추정          │               │ F 추정          │
│ (5-point)       │               │ (7-point)       │
└─────────────────┘               └─────────────────┘
       │                                    │
       └────────────┬───────────────────────┘
                    ▼
         ┌─────────────────┐
         │ H 추정          │
         │ (4-point)       │
         └─────────────────┘
                    │
                    ▼
         ┌─────────────────────────────────┐
         │ Inlier 비율 비교                 │
         │ E/F vs H                        │
         └─────────────────────────────────┘
                    │
       ┌────────────┼────────────┐
       ▼            ▼            ▼
   CALIBRATED   PLANAR/      DEGENERATE
                PANORAMIC
```



### 11.2 COLMAP 코드

In [ ]:
// src/colmap/estimators/two_view_geometry.cc
TwoViewGeometry EstimateTwoViewGeometry(
    const Camera& camera1,
    const std::vector<Eigen::Vector2d>& points1,
    const Camera& camera2,
    const std::vector<Eigen::Vector2d>& points2,
    FeatureMatches matches,
    const TwoViewGeometryOptions& options) {
  
  if (options.force_H_use) {
    return EstimateCalibratedHomography(...);
  } 
  else if (camera1.has_prior_focal_length && camera2.has_prior_focal_length) {
    // K를 알고 있음 → Essential Matrix 사용
    return EstimateCalibratedTwoViewGeometry(...);
  } 
  else {
    // K를 모름 → Fundamental Matrix 사용
    return EstimateUncalibratedTwoViewGeometry(...);
  }
}



---

## 11. RANSAC과 LO-RANSAC

### 11.1 RANSAC의 역할: 매칭이 아니라 "검증 + 모델 추정"

**흔한 오해:** "RANSAC이 특징점을 매칭한다"

**실제:** RANSAC은 **이미 매칭된 점들**에서 outlier를 제거하고 E를 추정합니다.

```
┌─────────────────────────────────────────────────────────────────────────┐
│  COLMAP 파이프라인에서 RANSAC의 위치                                     │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  1. Feature Extraction                                                  │
│        │  각 이미지에서 SIFT 특징점 추출                                  │
│        ▼                                                                │
│  2. Feature Matching  ← 여기서 "매칭"이 일어남!                          │
│        │  descriptor 유사도로 대응점 찾기                                │
│        │  결과: 100개 매칭 (하지만 30개는 잘못된 매칭 = outlier)          │
│        ▼                                                                │
│  3. Two-View Geometry + RANSAC  ← 여기서 RANSAC 사용!                   │
│        │  목적 1: outlier 제거 (잘못된 매칭 걸러내기)                     │
│        │  목적 2: E/F/H 행렬 추정 (기하학적 관계 계산)                    │
│        │  결과: 70개 inlier + Essential Matrix E                        │
│        ▼                                                                │
│  4. Pose Recovery, Triangulation, ...                                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```



| 단계 | 목적 | 방법 |
|------|------|------|
| **Feature Matching** | "어떤 점이 어떤 점과 대응되는가?" | Descriptor 유사도 비교 |
| **RANSAC** | "대응점 중 outlier 제거 + E 추정" | 기하학적 제약 검증 |

> **비유:**
> - Feature Matching = "이 사람과 저 사람이 같은 사람 같아요" (얼굴 유사도) → 100명 후보
> - RANSAC = "키, 나이, 지문 등으로 검증해서 진짜만 남기자" → 70명 확정 + 관계(E) 계산

### 11.2 RANSAC 알고리듬

**문제 상황:** Feature matching 결과에는 **outlier(잘못된 매칭)**가 섞여 있습니다.

```
전체 대응점 100개 중:
├── 70개: inlier (올바른 매칭) ✓
└── 30개: outlier (잘못된 매칭) ✗

문제: outlier가 섞인 데이터로 E를 추정하면 결과가 틀림!
```



**RANSAC의 핵심 아이디어:** "운 좋게 inlier만 뽑히면 좋은 모델이 나온다"

```
┌─────────────────────────────────────────────────────────────────────┐
│  RANSAC 알고리듬                                                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  입력: 대응점 100개 (inlier 70개 + outlier 30개 섞여있음)            │
│                                                                     │
│  for i = 1 to N번 반복:                                             │
│      ┌─────────────────────────────────────────────────────────┐   │
│      │ Step 1: 랜덤하게 최소 개수만큼 선택                       │   │
│      │         (E: 5점, F: 7점, H: 4점)                         │   │
│      │                                                         │   │
│      │     예: 100개 중 랜덤 5점 선택                            │   │
│      │         → 운 좋으면 5개 모두 inlier                       │   │
│      │         → 운 나쁘면 outlier 포함                         │   │
│      └─────────────────────────────────────────────────────────┘   │
│                          │                                         │
│                          ▼                                         │
│      ┌─────────────────────────────────────────────────────────┐   │
│      │ Step 2: 선택한 점들로 모델(E) 추정                        │   │
│      │                                                         │   │
│      │     5점 → 5-point algorithm → E 후보                     │   │
│      └─────────────────────────────────────────────────────────┘   │
│                          │                                         │
│                          ▼                                         │
│      ┌─────────────────────────────────────────────────────────┐   │
│      │ Step 3: 나머지 95개 점에 대해 Sampson Error 계산          │   │
│      │                                                         │   │
│      │     for each 나머지 점:                                  │   │
│      │         error = SampsonError(점, E)                     │   │
│      │         if error < 4.0:                                 │   │
│      │             inlier_count++                              │   │
│      └─────────────────────────────────────────────────────────┘   │
│                          │                                         │
│                          ▼                                         │
│      ┌─────────────────────────────────────────────────────────┐   │
│      │ Step 4: 이번 E가 역대 최고인가?                           │   │
│      │                                                         │   │
│      │     if inlier_count > best_inlier_count:                │   │
│      │         best_E = E                                      │   │
│      │         best_inlier_count = inlier_count                │   │
│      └─────────────────────────────────────────────────────────┘   │
│                                                                     │
│  출력: best_E (가장 많은 inlier를 설명하는 모델)                      │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```



**왜 이 방식이 작동하나?**

| 반복 | 랜덤 5점 선택 | 추정된 E | inlier 개수 | 비고 |
|------|--------------|----------|-------------|------|
| 1회 | ✗ outlier 포함 | 틀린 E | 15개 | |
| 2회 | ✗ outlier 포함 | 틀린 E | 22개 | |
| ... | ... | ... | ... | |
| 47회 | ✓ **모두 inlier** | **좋은 E** | **68개** | ← 최고! |
| ... | ... | ... | ... | |
| 100회 | ✗ outlier 포함 | 틀린 E | 18개 | |

> **직관:** 좋은 E는 **많은 점들을 잘 설명**합니다 (inlier가 많음). 나쁜 E는 **소수의 점만** 우연히 만족합니다. 따라서 inlier가 가장 많은 E를 선택하면 높은 확률로 올바른 모델입니다.

### 11.3 반복 횟수: 몇 번 반복해야 하는가?

**질문:** RANSAC을 몇 번 반복해야 좋은 모델을 찾을 확률이 충분히 높은가?

**핵심 아이디어:** "최소 1번은 **선택한 n개가 모두 inlier**인 경우가 나와야 한다"

```
예: 100개 점 중 70개가 inlier (70%), 30개가 outlier (30%)

랜덤으로 5개를 뽑을 때:
├── 5개 모두 inlier일 확률: 0.7⁵ ≈ 16.8%  ← 이 경우 좋은 E가 나옴!
├── 1개라도 outlier 포함: 83.2%           ← 이 경우 나쁜 E가 나옴
└── 여러 번 반복하면 언젠가 5개 모두 inlier인 경우가 나옴
```



**확률 분석 (단계별 유도):**

**Step 1: 한 개의 점이 inlier일 확률**

전체 대응점 중 inlier의 비율을 $w$라 하면:
$$P(\text{랜덤 1개가 inlier}) = w$$

예: 100개 중 70개가 inlier → $w = 0.7$

**Step 2: n개의 점이 모두 inlier일 확률**

n개의 점을 **독립적으로** 랜덤 선택한다고 가정하면:
$$P(\text{n개 모두 inlier}) = \underbrace{w \times w \times \cdots \times w}_{n\text{번}} = w^n$$

> **왜 곱셈인가?** 독립 사건의 동시 발생 확률은 각 확률의 곱입니다.
> - 1번째 점이 inlier: 확률 $w$
> - 2번째 점도 inlier: 확률 $w$ (1번째와 독립)
> - ...
> - n번째 점도 inlier: 확률 $w$
> - 모두 동시에 만족: $w \times w \times \cdots = w^n$

예: $w = 0.7$, $n = 5$ → $w^5 = 0.7^5 = 0.168$ (약 17%)

**Step 3: 한 번의 시도가 "실패"할 확률**

"실패" = n개 중 **하나라도** outlier가 포함됨 = "n개 모두 inlier"의 **여사건**

$$P(\text{실패}) = 1 - P(\text{성공}) = 1 - w^n$$

예: $1 - 0.168 = 0.832$ (약 83%)

**Step 4: N번 연속으로 "실패"할 확률**

N번의 시도가 **독립**이므로:
$$P(\text{N번 모두 실패}) = \underbrace{(1-w^n) \times (1-w^n) \times \cdots}_{N\text{번}} = (1-w^n)^N$$

> **왜 이것을 계산하나?** "최소 1번 성공"의 확률을 직접 계산하기 어렵기 때문에, **여사건**인 "N번 모두 실패"를 먼저 계산합니다.

**Step 5: 최소 1번 성공할 확률**

"최소 1번 성공" = "N번 모두 실패"의 **여사건**

$$P(\text{최소 1번 성공}) = 1 - P(\text{N번 모두 실패}) = 1 - (1-w^n)^N$$

**Step 6: 필요한 반복 횟수 N 유도**

"최소 1번 성공할 확률이 $p$ 이상"이 되려면:

$$
1 - (1 - w^n)^N \geq p
$$

양변 정리:

$$
(1 - w^n)^N \leq 1 - p
$$

로그를 취하면 ($(1-w^n) < 1$이므로 부등호 방향 유지):

$$
N \cdot \log(1 - w^n) \leq \log(1 - p)
$$

$$
N \geq \frac{\log(1 - p)}{\log(1 - w^n)}
$$

**최종 공식:**

$$
N = \frac{\log(1 - p)}{\log(1 - w^n)}
$$

| 기호 | 의미 | 예시 | 비고 |
|------|------|------|------|
| $n$ | **한 번에 선택하는 샘플 수** (소문자) | 5 (5-point) | 모델 추정에 필요한 최소 점 수 |
| $N$ | **총 반복 횟수** (대문자) | 145회 | 우리가 구하고 싶은 값 |
| $w$ | inlier 비율 | 0.5 (50%) | 전체 중 inlier의 비율 |
| $p$ | 원하는 성공 확률 | 0.99 (99%) | 보통 0.99 또는 0.999 |

> **$n$ vs $N$ 구분:**
> - $n$: 매 반복마다 **몇 개의 점**을 뽑는가? (E: 5개, F: 7개, H: 4개)
> - $N$: **몇 번 반복**해야 좋은 모델을 찾는가? (우리가 계산하는 값)
>
> $w^n$의 의미: $n$개를 뽑을 때 **모두 inlier일 확률** (각 점이 inlier일 확률 $w$가 $n$번 독립적으로 발생)

**구체적 예시:**

```
조건: inlier 비율 50%, 5-point, 99% 성공 원함

w⁵ = 0.5⁵ = 0.03125  (한 번에 성공할 확률 ≈ 3%)

N = log(1 - 0.99) / log(1 - 0.03125)
  = log(0.01) / log(0.96875)
  = -4.605 / -0.0317
  ≈ 145회

→ 145번 반복하면 99% 확률로 좋은 모델을 찾음
```



**N의 의미: 최소 vs 최대?**

| 관점 | 설명 |
|------|------|
| **이론적** | N은 "p 확률로 성공하기 위한 **최소** 반복 횟수" |
| **실제 구현** | N을 **최대** 반복 횟수로 사용 (early termination 가능) |

**COLMAP의 실제 동작:**

In [ ]:
// 실제로는 adaptive하게 동작
while (iteration < max_iterations) {
    // 모델 추정 및 inlier 계산
    ...
    
    // inlier 비율이 높으면 N을 재계산 (더 적게 반복해도 됨)
    if (inlier_ratio > previous_best) {
        max_iterations = ComputeNumIterations(inlier_ratio, ...);
    }
}



> **직관:** 처음에는 inlier 비율을 모르므로 보수적으로 많이 반복합니다. 좋은 모델을 찾으면 inlier 비율 추정치가 올라가고, 필요한 반복 횟수가 줄어들어 **일찍 종료**할 수 있습니다.

**inlier 비율에 따른 필요 반복 횟수 (p=0.99, n=5):**

| inlier 비율 $w$ | $w^5$ | 필요 반복 $N$ |
|----------------|-------|---------------|
| 90% | 0.59 | 8회 |
| 70% | 0.17 | 26회 |
| 50% | 0.03 | 145회 |
| 30% | 0.002 | 2,011회 |

> **결론:** Outlier가 많을수록, 최소 샘플이 많을수록 반복 횟수가 **기하급수적으로** 증가합니다. 이것이 RANSAC이 outlier가 매우 많은 경우(>50%) 비효율적인 이유입니다.

### 11.4 LO-RANSAC (Local Optimization)

**LO-RANSAC = RANSAC + Local Optimization**

LO-RANSAC은 기본 RANSAC에 **추가 정제 단계**를 더한 개선된 버전입니다.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│   LO-RANSAC = RANSAC 전체 + Local Optimization 추가                     │
│                                                                         │
│   ┌─────────────────────────────────────────────────────────────────┐   │
│   │  RANSAC 부분 (그대로 수행)                                       │   │
│   │                                                                 │   │
│   │   for i = 1 to N:                                               │   │
│   │       1. 랜덤 5점 선택                                           │   │
│   │       2. 5점으로 E 추정                                          │   │
│   │       3. 모든 점에 대해 Sampson Error 계산                        │   │
│   │       4. inlier 개수 카운트                                      │   │
│   │                                                                 │   │
│   │       if 역대 최고 기록 갱신:                                     │   │
│   │           best_E = E                                            │   │
│   │           ┌─────────────────────────────────────────────────┐   │   │
│   │           │ ★ Local Optimization (LO-RANSAC만의 추가 단계)  │   │   │
│   │           │                                                 │   │   │
│   │           │   inlier 68개 전체로 E 재추정                    │   │   │
│   │           │   → 더 좋으면 best_E 갱신                        │   │   │
│   │           └─────────────────────────────────────────────────┘   │   │
│   │                                                                 │   │
│   └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│   return best_E                                                         │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```



**비유로 이해하기:**

| | RANSAC | LO-RANSAC |
|---|---|---|
| **비유** | 시험 보고 바로 제출 | 시험 보고 **검토 후** 제출 |
| **과정** | 5점으로 E 추정 → 끝 | 5점으로 E 추정 → **inlier로 다시 추정** → 끝 |

**왜 LO가 더 정확한가?**

```
RANSAC:
    랜덤 5점 → E 추정 → "이게 최고!" → 바로 저장
                         ↑
                    5점의 노이즈가 100% 반영됨

LO-RANSAC:
    랜덤 5점 → E 추정 → "이게 최고!" → inlier 68개로 다시 추정 → 저장
                                                ↑
                                         68점의 노이즈가 평균화됨
```



**왜 LO-RANSAC이 더 빠른가?**

얼핏 보면 "추가 계산 = 더 느림"이라고 생각할 수 있지만, **전체적으로는 LO-RANSAC이 더 빠릅니다**:

```
RANSAC의 반복 횟수 공식: N = log(1-p) / log(1-w^n)

핵심: w(inlier 비율)가 올라가면 N이 줄어듦!

┌─────────────────────────────────────────────────────────────────────────┐
│  RANSAC:                                                                │
│                                                                         │
│    5점으로 추정한 E는 노이즈로 인해 "약간 틀림"                           │
│    → 실제 inlier 70개 중 60개만 검출 (w = 0.6)                           │
│    → N = 145회 필요                                                     │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│  LO-RANSAC:                                                             │
│                                                                         │
│    5점으로 추정 → 60개 inlier 검출 → 60개로 재추정 → 더 정확한 E          │
│    → 70개 전부 검출 가능 (w = 0.7)                                       │
│    → N = 26회로 충분! (adaptive termination)                            │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```



| | RANSAC | LO-RANSAC |
|---|---|---|
| **개별 반복 비용** | 낮음 | 높음 (재추정 비용) |
| **필요한 반복 횟수** | 많음 | **적음** (일찍 종료) |
| **전체 시간** | 느림 | **빠름** |
| **정확도** | 낮음 | **높음** |

> **핵심 통찰:** LO로 모델을 정제하면 **더 많은 inlier를 찾음** → **w 추정치 상승** → **N 감소** → **일찍 종료**

**정리:**

| 질문 | 답 |
|------|-----|
| RANSAC 없이 LO-RANSAC 가능? | **불가능!** LO-RANSAC은 RANSAC을 포함합니다 |
| LO-RANSAC이 더 좋은가? | **예!** 더 빠르고 더 정확함 |
| 차이점은? | "최고 기록 갱신 시" 추가 정제 단계가 있음 |
| 단점은? | 구현이 약간 더 복잡함 (그러나 성능은 우월) |

> **결론:** 현대 컴퓨터 비전에서 LO-RANSAC은 사실상 표준입니다. OpenCV의 `findHomography`, `findEssentialMat` 등도 내부적으로 LO-RANSAC 변형(USAC_MAGSAC 등)을 사용합니다.

**COLMAP에서의 사용:**

COLMAP은 기본적으로 **모든** 기하 추정에 LO-RANSAC을 사용합니다:

In [ ]:
// src/colmap/estimators/two_view_geometry.cc

// Essential Matrix 추정 시
LORANSAC<EssentialMatrixFivePointEstimator, EssentialMatrixFivePointEstimator>
    E_ransac(E_ransac_options);

// Fundamental Matrix 추정 시
LORANSAC<FundamentalMatrixSevenPointEstimator, FundamentalMatrixEightPointEstimator>
    F_ransac(F_ransac_options);

// Homography 추정 시
LORANSAC<HomographyMatrixEstimator, HomographyMatrixEstimator>
    H_ransac(H_ransac_options);



즉, `colmap feature_matcher` 또는 `colmap mapper`를 실행하면 **자동으로 LO-RANSAC**이 사용됩니다. 별도 설정 없이 3DGS 데이터 준비 시에도 적용됩니다.

**여기서 "모델"이란?**

| 문맥 | 모델 | 의미 |
|------|------|------|
| E 추정 | Essential Matrix $\mathbf{E}$ | 3×3 행렬 (두 카메라 간 상대 포즈 인코딩) |
| F 추정 | Fundamental Matrix $\mathbf{F}$ | 3×3 행렬 |
| H 추정 | Homography $\mathbf{H}$ | 3×3 행렬 |

**Local Optimization이 더 정확한 이유:**

```
기본 RANSAC (5점만 사용):

    100개 점 중 랜덤 5점 선택
              │
              ▼
    ┌─────────────────────┐
    │  5점으로 E 추정      │  ← 최소 개수만 사용
    │  (정확히 맞는 해)     │     (노이즈에 민감)
    └─────────────────────┘
              │
              ▼
         inlier 68개 발견
              │
              ▼
         이 E를 최종 결과로 사용 ← 5점의 정보만 반영됨


LO-RANSAC (모든 inlier 사용):

    100개 점 중 랜덤 5점 선택
              │
              ▼
    ┌─────────────────────┐
    │  5점으로 E 추정      │
    └─────────────────────┘
              │
              ▼
         inlier 68개 발견
              │
              ▼
    ┌─────────────────────────────────────┐
    │  68개 inlier 전체로 E 재추정         │  ← Local Optimization!
    │  (overdetermined → least squares)  │     (68개 정보 반영)
    └─────────────────────────────────────┘
              │
              ▼
         더 정확한 E를 최종 결과로 사용
```



**왜 더 정확한가?**

| | 5점으로 추정 | 68점으로 추정 |
|---|---|---|
| **방정식 수** | 5개 (정확히 맞음) | 68개 (과결정) |
| **해법** | 정확해 (unique solution) | 최소자승해 (least squares) |
| **노이즈 영향** | 5점의 노이즈가 100% 반영 | 68점의 노이즈가 **평균화** |
| **결과** | 노이즈에 민감 | **노이즈에 강건** |

> **핵심:** 5점은 E를 추정하기 위한 **최소 개수**입니다. 더 많은 점을 사용하면 개별 점의 노이즈가 서로 상쇄되어 **평균적으로 더 좋은 E**를 얻습니다. 이것이 "overdetermined system의 least squares 해"가 robust한 이유입니다.

**수학적 관점:**

$$
\text{5점: } \quad A_{5 \times 9} \cdot \text{vec}(E) = 0 \quad \text{(정확히 풀림)}
$$

$$
\text{68점: } \quad A_{68 \times 9} \cdot \text{vec}(E) \approx 0 \quad \text{(최소자승으로 풀림)}
$$

68개 방정식을 **동시에 가장 잘 만족**하는 E를 찾습니다. SVD를 사용하면 $\|A \cdot \text{vec}(E)\|^2$를 최소화하는 E를 구할 수 있습니다.

---

## 12. Incremental Reconstruction 개요

Two-View Geometry(2~11장)로 초기 복원이 완료되면, **이미지를 하나씩 추가**하며 점진적으로 3D 복원을 확장합니다.

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Incremental Reconstruction 흐름                                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  [Two-View로 초기화]                                                     │
│        │                                                                │
│        │  초기 2장 → E 추정 → R,t 분해 → Triangulation                   │
│        │  결과: 카메라 2대 포즈 + 초기 3D 점들                            │
│        ▼                                                                │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  while (등록 안 된 이미지 존재):                                  │   │
│  │                                                                 │   │
│  │    ① 새 이미지 선택                                              │   │
│  │       → 이미 복원된 3D 점들과 가장 많이 매칭되는 이미지            │   │
│  │                                                                 │   │
│  │    ② PnP로 카메라 포즈 추정                                       │   │
│  │       → 3D-2D 대응점을 이용해 새 카메라의 위치/방향 계산          │   │
│  │                                                                 │   │
│  │    ③ Triangulation                                               │   │
│  │       → 새 카메라와 기존 카메라들 사이에서 새 3D 점 복원          │   │
│  │                                                                 │   │
│  │    ④ Bundle Adjustment                                           │   │
│  │       → 지금까지의 모든 카메라 + 모든 3D 점을 동시에 최적화       │   │
│  │       → 누적 오차 제거                                           │   │
│  │                                                                 │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│        │                                                                │
│        ▼                                                                │
│  [출력: 모든 카메라 포즈 + Sparse Point Cloud]                           │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```



### 12.1 PnP (Perspective-n-Point)

이미 알려진 3D 점들을 이용해 **새 카메라의 포즈**를 추정합니다.

#### 왜 E 행렬이 아니라 PnP를 사용하는가?

| | E 행렬 (Two-View) | PnP |
|---|-------------------|-----|
| **입력** | 2D-2D 대응 | **3D**-2D 대응 |
| **출력** | 상대 포즈 | **절대** 포즈 |
| **Scale** | 알 수 없음 (방향만) | 기존 모델과 일치 |
| **용도** | 초기 두 카메라만 | **3번째 카메라부터** |
| **좌표계** | 새로운 좌표계 생성 | 기존 좌표계에 통합 |

```
[E 행렬만 사용하면 문제 발생]

cam1 ─E₁₂─ cam2 ─E₂₃─ cam3 ─E₃₄─ cam4
      ↓          ↓          ↓
   scale₁    scale₂    scale₃   ← 각각 다른 scale!
   
결과: 3D 모델이 일관되지 않음 (drift 발생)


[PnP 사용 - 올바른 방법]

cam1 ─E₁₂─ cam2    cam3    cam4
      ↓      ↓       ↓       ↓
   scale₁   PnP     PnP     PnP
            ↓       ↓       ↓
        기존 3D 점 기반으로 절대 위치 계산
         
결과: 모든 카메라가 같은 좌표계, scale 일관
```



> **핵심:** E 행렬은 **상대 포즈**만 제공하므로 기존 좌표계와 연결 불가. PnP는 **이미 존재하는 3D 점**을 기준으로 **절대 포즈**를 계산하므로 기존 모델에 일관되게 통합됩니다.

#### 문제 설정

```
        ● X₁ (알고 있음)      ● X₂ (알고 있음)
         \                   /
          \                 /
           \   ● X₃        /
            \   |         /
             \  |        /
              \ |       /
               \|      /
                ○ ← 카메라 (R, t) = ? 찾고 싶음!
                |
        ┌───────┴───────┐
        │  x₁  x₂  x₃   │  ← 2D 투영 (알고 있음)
        │     image     │
        └───────────────┘
```



| 항목 | 내용 |
|------|------|
| **입력** | n개의 3D-2D 대응 $(\mathbf{X}_i, \mathbf{x}_i)$ |
| **출력** | 카메라 포즈 $[R\|t]$ |
| **최소 점 수** | 3점 (P3P) |
| **COLMAP** | P3P + EPnP + LO-RANSAC |

#### P3P의 직관

3개의 3D-2D 대응이 주어지면:

```
           X₁●────────────●X₂
              \    d₁₂   /
               \        /
            d₁₃ \      / d₂₃  ← 3D 점 사이 거리 (알고 있음)
                 \    /
                  \  /
                   ●X₃

    카메라에서 본 광선들:
         ray₁  ray₂  ray₃
           \    |    /
            \   |   /
             \  |  /
              \ | /
               \|/
                ○ camera
```



1. 각 2D 점에서 광선을 쏨
2. 3D 점들 사이의 **거리**는 알고 있음
3. 광선 위에서 거리 조건을 만족하는 점 찾기 → **4차 방정식, 최대 4개 해**

#### COLMAP 코드

In [ ]:
// src/colmap/estimators/pose.cc
// P3P로 초기 추정, EPnP로 Local Optimization

LORANSAC<P3PEstimator, EPNPEstimator> ransac(
    options.ransac_options,
    P3PEstimator(img_from_cam_func),
    EPNPEstimator(img_from_cam_func));

auto report = ransac.Estimate(points2D_with_rays, points3D);

if (report.success) {
  // report.model: 3x4 행렬 [R|t]
  *cam_from_world = Rigid3d(
      Eigen::Quaterniond(report.model.leftCols<3>()),  // R
      report.model.col(3)                               // t
  );
  *num_inliers = report.support.num_inliers;
}



> **참고:** Two-View E 추정과 동일한 LO-RANSAC 패턴입니다:
> - **P3P** (3점): 빠른 모델 추정 → RANSAC sampling에 사용
> - **EPnP** (모든 inlier): 정밀한 Local Optimization

### 12.2 Triangulation (삼각측량)

새 카메라가 추가되면, 기존 카메라들과 **새 3D 점**을 생성합니다.

#### 문제 설정

```
              ● X = ? (찾고 싶은 3D 점)
             /\
            /  \
       ray1/    \ray2
          /      \
         /        \
        ○──────────○
       cam2       cam4 (새로 추가됨)
      (R₂,t₂)    (R₄,t₄)  ← 포즈 알고 있음!
        ↓          ↓
       x₂         x₄  ← 관측된 2D 점 (알고 있음)
```



| 항목 | 내용 |
|------|------|
| **입력** | 2개 이상의 카메라 포즈 + 각 카메라의 2D 관측점 |
| **출력** | 3D 점 좌표 $\mathbf{X}$ |
| **방법** | 광선의 교점 (최소자승) |

#### 왜 Triangulation이 필요한가?

```
cam4에서 검출된 feature들:
┌─────────────────────┐
│  ●  ●  ○  ●  ○  ●  │   ● = 기존 3D 점과 매칭됨 (PnP에 사용)
│  ○  ●  ●  ○  ●  ○  │   ○ = 새로운 feature! (아직 3D 점 없음)
│  ●  ○  ●  ○  ●  ●  │
└─────────────────────┘

새 feature들은 다른 카메라에서도 보임:

        cam2                    cam4
    ┌─────────┐             ┌─────────┐
    │    ○a   │             │  ○a     │   ← 같은 점!
    │  ○b     │             │      ○b │
    └─────────┘             └─────────┘
         \                     /
          \    ● NEW!         /    ← Triangulation으로
           \    3D점        /        새 3D 점 생성
            \              /
```



#### 기하학적 해법

- **이상적:** 두 광선이 정확히 교차
- **현실:** 노이즈로 인해 광선이 만나지 않음 (skew lines)
- **해결:** 두 광선에 **가장 가까운 점**을 최소자승으로 찾음

$$
A\mathbf{X} = 0 \quad \rightarrow \quad \mathbf{X} = \text{SVD}(A)\text{의 마지막 열}
$$

### 12.3 Bundle Adjustment (BA)

**모든 카메라 포즈**와 **모든 3D 점**을 **동시에** 최적화합니다.

#### 왜 BA가 필요한가?

```
PnP와 Triangulation은 각각 독립적으로 계산됨:

PnP:         "3D 점 좌표가 정확하다"고 가정 → 카메라 포즈 계산
Triangulation: "카메라 포즈가 정확하다"고 가정 → 3D 점 계산

⚠️ 문제: 둘 다 약간의 오차가 있음!
         이미지를 추가할수록 오차가 누적됨

BA: 모든 것을 동시에 고려하여 "전체적으로 가장 일관된" 해를 찾음
```



#### 목적 함수

$$
\min_{R_i, t_i, X_j} \sum_i \sum_j \| \mathbf{x}_{ij} - \pi(R_i, t_i, \mathbf{X}_j) \|^2
$$

| 기호 | 의미 |
|------|------|
| $\mathbf{x}_{ij}$ | 카메라 $i$에서 관측된 점 $j$의 2D 좌표 |
| $\pi(R_i, t_i, \mathbf{X}_j)$ | 3D 점 $\mathbf{X}_j$를 카메라 $i$로 **재투영**한 2D 좌표 |
| $\|\cdot\|^2$ | **Reprojection Error** (재투영 오차) |

```
                3D 점 Xⱼ
                   ●
                  /|\ 
                 / | \
                /  |  \
    관측: x₁ⱼ ○   |   ○ 관측: x₂ⱼ    ← 실제 관측된 2D 점
               ↑   |   ↑
          오차 │   |   │ 오차        ← 이 오차들의 합을 최소화!
               ↓   |   ↓
    재투영: π₁ ●   |   ● 재투영: π₂   ← 3D 점을 카메라로 투영한 위치
                  cam₁ cam₂
```



#### 최적화 전후 비교

```
            [최적화 전]                         [최적화 후]

        ● 3D점 (약간 틀린 위치)              ● 3D점 (정확한 위치)
       /|\                                 /|\
      / | \                               / | \
     /  |  \   재투영 오차 큼              /  |  \   재투영 오차 최소
    ○   ○   ○   (2D에서 빗나감)           ○   ○   ○   (2D에서 정확히 맞음)
  cam1 cam2 cam3                      cam1 cam2 cam3
  (포즈 약간 틀림)                     (포즈 정확)
```



#### COLMAP 코드

In [ ]:
// src/colmap/sfm/incremental_mapper.cc
// 새 이미지 등록 후 Local Bundle Adjustment 수행

std::unique_ptr<BundleAdjuster> bundle_adjuster =
    CreateDefaultBundleAdjuster(ba_options, ba_config, *reconstruction_);

const auto summary = bundle_adjuster->Solve();

report.num_adjusted_observations = summary->num_residuals / 2;

In [ ]:
// src/colmap/estimators/bundle_adjustment.h
// Bundle Adjustment 설정

class BundleAdjustmentConfig {
 public:
  void AddImage(image_t image_id);        // 최적화할 이미지 추가
  void AddVariablePoint(point3D_t);       // 최적화할 3D 점 추가
  void AddConstantPoint(point3D_t);       // 고정할 3D 점 (최적화 안 함)
  
  size_t NumResiduals(const Reconstruction&) const;  // 총 residual 개수
};



> **참고:** BA는 **Levenberg-Marquardt** 알고리듬으로 비선형 최적화를 수행하며, COLMAP은 **Ceres Solver**를 사용합니다.

### 12.4 전체 흐름 요약

```
┌─────────────────────────────────────────────────────────────────┐
│                  한 이미지 추가 사이클                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  [현재 상태]                                                     │
│  - N개 카메라 (포즈 알고 있음)                                    │
│  - M개 3D 점 (좌표 알고 있음)                                     │
│                                                                 │
│      │                                                          │
│      ▼                                                          │
│  ┌─────────────────────────────────────┐                        │
│  │ 1. PnP                              │                        │
│  │    목적: 새 카메라 포즈 계산          │                        │
│  │    입력: 기존 3D 점 + 새 이미지 2D 점 │                        │
│  │    출력: 새 카메라의 (R, t)          │                        │
│  │    결과: N → N+1개 카메라            │                        │
│  └─────────────────────────────────────┘                        │
│      │                                                          │
│      ▼                                                          │
│  ┌─────────────────────────────────────┐                        │
│  │ 2. Triangulation                    │                        │
│  │    목적: 새 3D 점 생성               │                        │
│  │    입력: 카메라 포즈들 + 2D 대응점    │                        │
│  │    출력: 새 3D 점 좌표               │                        │
│  │    결과: M → M+K개 3D 점             │                        │
│  └─────────────────────────────────────┘                        │
│      │                                                          │
│      ▼                                                          │
│  ┌─────────────────────────────────────┐                        │
│  │ 3. Bundle Adjustment                │                        │
│  │    목적: 전체 정밀화                  │                        │
│  │    입력: 모든 카메라 + 모든 3D 점     │                        │
│  │    출력: 정밀화된 포즈 + 3D 점        │                        │
│  │    결과: 누적 오차 제거               │                        │
│  └─────────────────────────────────────┘                        │
│      │                                                          │
│      ▼                                                          │
│  [다음 이미지로 반복]                                             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```



#### 숫자 예시

| 단계 | 카메라 수 | 3D 점 수 | 설명 |
|------|----------|----------|------|
| 초기 (Two-View) | 2 | 100 | E 행렬 + Triangulation |
| cam3 추가 후 | 3 | 130 | PnP(50점 사용) + Tri(+30점) + BA |
| cam4 추가 후 | 4 | 155 | PnP(60점 사용) + Tri(+25점) + BA |
| ... | ... | ... | ... |
| 최종 (100장) | 100 | ~10,000 | 전체 복원 완료 |

#### 각 단계의 역할

```
PnP          →  "카메라 포즈"  →  Triangulation에 필요
Triangulation →  "3D 점"      →  다음 PnP에 필요  
BA           →  "정밀화"      →  누적 오차 방지

     ┌──────────────────────────────────┐
     │                                  │
     ▼                                  │
   [PnP] ──→ [Triangulation] ──→ [BA] ──┘
     │              │              │
  카메라 추가    3D 점 추가     전체 정밀화
```



---

## 13. 참고 자료

**교과서:**
- Hartley & Zisserman, "Multiple View Geometry in Computer Vision" (MVG) - 다중시점 기하학의 바이블

**논문:**
- Nistér, "An efficient solution to the five-point relative pose problem" (CVPR 2003) - 5-point 알고리듬

**온라인:**
- [First Principles of Computer Vision - Multiview Geometry](https://www.youtube.com/playlist?list=PL2zRqk16wsdp8KbDfHKvPYNGF2L-zQASc)
- [OpenCV Epipolar Geometry Tutorial](https://docs.opencv.org/4.x/da/de9/tutorial_py_epipolar_geometry.html)